<a href="https://colab.research.google.com/github/rakasiwisurya/pgaibllm-lessons/blob/main/Latihan_Membuat_Multi_Agent_Menggunakan_CrewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Library: Menyiapkan Lingkungan Agent

In [1]:
!pip install -q -U crewai litellm langchain-groq duckduckgo-search ddgs apscheduler "pydantic[email]" langchain-community
!pip install fastapi fastapi-sso uvicorn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import nest_asyncio
nest_asyncio.apply()

/tmp/ipykernel_1094/265110612.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


# Menghubungkan Sistem ke Dunia Luar

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["OPENAI_API_KEY"] = "sk-dummy"

# Menentukan Model LLM yang Digunakan

In [4]:
my_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.0,
    api_key=userdata.get('GROQ_API_KEY')
)

# Tool: Cara Agent Meng-explore Dunia Luar

In [5]:
search_instance = DuckDuckGoSearchRun()
@tool("web_search")
def web_search(query: str):
    """Bermanfaat untuk mencari data real-time di internet.
    Gunakan kata kunci pencarian yang spesifik dan singkat."""
    return search_instance.run(query)

# Membagi Peran Agent: Siapa Melakukan Apa

## Researcher: Mencari dan Menyaring

In [6]:
researcher = Agent(
    role='Lead Tech Researcher',
    goal='Mencari dan merangkum 3 tren utama {topic} di tahun 2026',
    backstory='''Kamu adalah peneliti senior di biro teknologi global.
    Keahlianmu adalah menemukan informasi valid dan mengubahnya menjadi laporan poin-poin.''',
    tools=[web_search],
    llm=my_llm,
    verbose=True,
    max_iter=3
)

## Writer: Mengubah Data Menjadi Cerita

In [14]:
writer = Agent(
    role='Tech Content Strategist',
    goal='Menyusun artikel blog edukatif berdasarkan hasil riset teknis',
    backstory='''Kamu adalah penulis blog teknologi yang ahli.
    Gaya bahasamu ramah, profesional, dan mudah dipahami, mirip gaya penulisan Dicoding.''',
    llm=my_llm,
    verbose=True
)

# Task: Cara Agent Saling Menunggu dan Meneruskan Hasil

In [15]:
task_research = Task(
    description='Gunakan tool web_search untuk riset {topic}. Identifikasi 3 inovasi penting untuk tahun 2026.',
    expected_output='Laporan riset berisi 3 poin inovasi utama.',
    agent=researcher
)

task_write = Task(
    description='Buat artikel blog 3 paragraf berdasarkan laporan riset dari researcher.',
    expected_output='Artikel blog lengkap dalam format Markdown.',
    agent=writer
)

# Crew: Koordinator yang Menjaga Alur Tetap Rapi

In [16]:
my_crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    process=Process.hierarchical,
    manager_llm=my_llm,
    verbose=True
)

# Menjalankan Sistem

In [17]:
if __name__ == "__main__":
    print("--- [CrewAI Multi-Agent Execution Started] ---")
    try:
        # Menjalankan workflow dengan input topik tertentu
        result = my_crew.kickoff(inputs={'topic': 'Agentic AI Workflow'})
        print("\n--- [Hasil Akhir] ---")
        print(result)
    except Exception as e:
        print(f"Terjadi error: {e}")

--- [CrewAI Multi-Agent Execution Started] ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 23c37768-8805-48bb-9322-a4c8403eb147                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Gunakan tool web_search untuk riset Agentic AI Workflow. Identifikasi 3 inovasi penting untuk tahun      │
│  2026.                                                                                                          │
│  ID: 3aff8b13-4301-44a4-b694-fbc625cf181f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Gunakan tool web_search untuk riset Agentic AI Workflow. Identifikasi 3 inovasi penting untuk tahun      │
│  2026.                                                                                                          │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Terjadi error: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯